# SLIIT IT3051 - EV Battery PHM Dual-Task System
## Week 2 - Task 1: Linear & Regularized Regression Models (RUL Prediction)
### **Assigned Member: Person A**

---
### Overview & Objectives:
Evaluate Linear Regression baselines, Ridge (L2) and Lasso (L1) regularization, alpha tuning via cross-validation, and feature sparsity analysis.


### 0. Environment Setup & Data Partition Loading
Execute this cell to load the verified preprocessed matrices.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

def load_data():
    dataset_path = 'ev battery_failure  Dataset.csv'
    for p in [dataset_path, f'../{dataset_path}', f'../../{dataset_path}']:
        if os.path.exists(p):
            dataset_path = p
            break
            
    df = pd.read_csv(dataset_path)
    
    # 7 Domain Features
    df['temperature_spread'] = df['cell_temperature_max'] - df['cell_temperature_avg']
    df['efficiency_gap'] = (df['charge_efficiency'] - df['discharge_efficiency']).abs()
    df['health_loss_interaction'] = (df['battery_health_percent'] * df['capacity_loss_percent']) / 100.0
    df['resistance_per_1000_cycles'] = (df['internal_resistance'] / (df['cycle_count'] + 1.0)) * 1000.0
    df['c_rate_proxy'] = df['average_charge_power_kw'] / (df['battery_capacity_kwh'] + 1e-5)
    df['cell_voltage_spread'] = df['cell_voltage_std'] / (df['cell_voltage_avg'] + 1e-5)
    df['stress_index'] = df['aggressive_acceleration_score'] * df['hard_braking_score']
    
    id_cols = ['vehicle_id', 'battery_serial']
    target_rul = 'predicted_remaining_life_cycles'
    target_fail = 'battery_failure'
    
    excluded = id_cols + [target_rul, target_fail]
    features = [c for c in df.columns if c not in excluded]
    
    num_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df[features].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    
    num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    
    # Task 1 Splits (RUL)
    df_t1 = df[df[target_rul].notnull()].copy()
    X1_tr, X1_te, y1_train, y1_test = train_test_split(df_t1[features], df_t1[target_rul], test_size=0.20, random_state=42)
    ct1 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X1_train = ct1.fit_transform(X1_tr)
    X1_test = ct1.transform(X1_te)
    
    # Task 2 Splits (Failure)
    X2_tr, X2_te, y2_train, y2_test = train_test_split(df[features], df[target_fail], test_size=0.20, random_state=42, stratify=df[target_fail])
    ct2 = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)], verbose_feature_names_out=False)
    X2_train = ct2.fit_transform(X2_tr)
    X2_test = ct2.transform(X2_te)
    
    feature_names = ct1.get_feature_names_out()
    return (X1_train, X1_test, y1_train, y1_test), (X2_train, X2_test, y2_train, y2_test), feature_names

print("Data Loader initialized successfully!")



In [ ]:
(X1_train, X1_test, y1_train, y1_test), _, feature_names = load_data()
print(f'Task 1 Train Shape: {X1_train.shape} | Test Shape: {X1_test.shape}')


### Task 1: Baseline Models: Dummy Regressor & Ordinary Least Squares
Train a baseline Dummy Regressor (predicting the training mean) and a standard Multiple Linear Regression model. Evaluate both using R^2, RMSE, and MAE.


In [ ]:
# TODO: Import DummyRegressor, LinearRegression, mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: Fit DummyRegressor(strategy='mean') on X1_train, y1_train

# TODO: Fit LinearRegression() on X1_train, y1_train

# TODO: Evaluate train & test R^2, RMSE, MAE for both models



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What is the benchmark RMSE and MAE achieved by the baseline Dummy Regressor?
- How much does Linear Regression improve over the naive mean benchmark?


### Task 2: Regularized Regression: Ridge (L2) & Lasso (L1)
Train Ridge regression to mitigate multicollinearity, and Lasso regression to enforce sparsity (automated feature selection).


In [ ]:
# TODO: Import Ridge and Lasso from sklearn.linear_model
from sklearn.linear_model import Ridge, Lasso

# TODO: Train Ridge(alpha=1.0) and evaluate R^2, RMSE, MAE

# TODO: Train Lasso(alpha=0.1) and evaluate R^2, RMSE, MAE

# TODO: Count how many feature coefficients were shrunk to exactly 0 by Lasso



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Did Ridge regression improve test RMSE compared to unregularized Linear Regression? Why?
- How many feature weights did Lasso shrink to zero? What does this tell us about redundant features?


### Task 3: Hyperparameter Tuning: Alpha Search via Cross-Validation
Perform 5-fold cross-validation across a grid of alpha values (e.g. 10^-3 to 10^3) using RidgeCV / LassoCV or GridSearchCV to find the optimal regularization strength.


In [ ]:
# TODO: Implement RidgeCV or GridSearchCV for Ridge tuning
from sklearn.linear_model import RidgeCV, LassoCV

# TODO: Fit and extract optimal alpha parameter

# TODO: Plot test performance vs alpha values



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- What was the optimal alpha value discovered by cross-validation?
- Does high regularization cause underfitting? Explain the bias-variance trade-off observed.


### Task 4: Linear Model Comparison Summary & Viva Defense
Compile a comparison table summarizing Dummy, Linear Regression, Tuned Ridge, and Tuned Lasso across Train/Test R^2, RMSE, and MAE.


In [ ]:
# TODO: Construct a pandas DataFrame summarizing all linear regression results



**Analysis & Viva Preparation Notes (Fill this in after running code):**
- Is there evidence of overfitting in the linear models (Train R^2 vs Test R^2)?
- Why is Linear Regression insufficient as our final production model for battery RUL?
